<a href="https://colab.research.google.com/github/Kripa-Garg/ipl-analytics-dashboard-/blob/main/IPL_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import zipfile
import os

# Define the path to the zip file and the extraction directory
zip_file_path = '/content/drive/MyDrive/IPL_Dataset.zip'
extract_dir = '/content/IPL_Dataset'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Files unzipped to: {extract_dir}")
print("Contents of the unzipped directory:")
print(os.listdir(extract_dir))

Files unzipped to: /content/IPL_Dataset
Contents of the unzipped directory:
['matches.csv', 'deliveries.csv']


In [4]:
matches = pd.read_csv('/content/IPL_Dataset/matches.csv')
deliveries = pd.read_csv('/content/IPL_Dataset/deliveries.csv')

# explore matches
print("Matches shape:", matches.shape)
print("\nColumns:", matches.columns.tolist())
print("\nFirst 3 rows:")
matches.head(3)

Matches shape: (1095, 20)

Columns: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

First 3 rows:


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar


In [5]:
# check for missing values
print("Matches nulls:")
print(matches.isnull().sum())

print("\nDeliveries shape:", deliveries.shape)
print("\nDeliveries nulls:")
print(deliveries.isnull().sum())

Matches nulls:
id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

Deliveries shape: (260920, 17)

Deliveries nulls:
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder  

In [6]:
# how many seasons?
print("Seasons:", sorted(matches['season'].unique()))

# how many unique teams?
all_teams = pd.concat([matches['team1'], matches['team2']]).unique()
print("\nAll teams:", sorted(all_teams))
print("Total unique teams:", len(all_teams))

Seasons: ['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024']

All teams: ['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
Total unique teams: 19


In [7]:
# explore deliveries
print("Deliveries shape:", deliveries.shape)
print("\nColumns:", deliveries.columns.tolist())
print("\nSample rows:")
deliveries.head(3)

Deliveries shape: (260920, 17)

Columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

Sample rows:


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN


In [8]:
# standardise team names — common inconsistencies in this dataset
team_name_fix = {
    'Rising Pune Supergiant':  'Rising Pune Supergiants',
    'Delhi Daredevils':        'Delhi Capitals',
    'Deccan Chargers':         'Sunrisers Hyderabad',
    'Kings XI Punjab':         'Punjab Kings',
    'Pune Warriors':           'Rising Pune Supergiants',
}

# apply to all team columns in matches
for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_name_fix)

# apply to deliveries too
for col in ['batting_team','bowling_team']:
    deliveries[col] = deliveries[col].replace(team_name_fix)

# verify — should now be fewer unique teams
print("Teams after fix:", len(pd.concat([matches['team1'],matches['team2']]).unique()))

Teams after fix: 14


In [9]:
# see how many matches have no result
print("Null winners:", matches['winner'].isnull().sum())

# drop them — these are abandoned or no-result matches
# we can't use them for win/loss analysis
matches_clean = matches.dropna(subset=['winner']).copy()
print("Matches after dropping nulls:", len(matches_clean))

Null winners: 5
Matches after dropping nulls: 1090


In [10]:
# did the toss winner also win the match?
matches_clean['toss_match_win'] = (
    matches_clean['toss_winner'] == matches_clean['winner']
)

# did the team batting second win? (chose to field OR team2 batted)
matches_clean['bat_first_team'] = matches_clean.apply(
    lambda r: r['team1'] if r['toss_decision']=='field'
              else r['team2'], axis=1
)
matches_clean['chasing_team_won'] = (
    matches_clean['bat_first_team'] != matches_clean['winner']
)

# win margin type
matches_clean['win_type'] = matches_clean.apply(
    lambda r: 'by runs' if r['result'] == 'runs'
              else 'by wickets', axis=1
)

print(matches_clean[['winner','toss_match_win','chasing_team_won','win_type']].head())

                        winner  toss_match_win  chasing_team_won    win_type
0        Kolkata Knight Riders           False              True     by runs
1          Chennai Super Kings            True             False     by runs
2               Delhi Capitals           False              True  by wickets
3  Royal Challengers Bangalore           False             False  by wickets
4        Kolkata Knight Riders           False              True  by wickets


In [11]:
# merge season and venue from matches into deliveries
# this lets us filter deliveries by season or venue later
deliveries_full = deliveries.merge(
    matches_clean[['id','season','venue','winner']],
    left_on='match_id',
    right_on='id',
    how='left'
)

print("Merged shape:", deliveries_full.shape)
print(deliveries_full.columns.tolist())

Merged shape: (260920, 21)
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'id', 'season', 'venue', 'winner']


In [12]:
# parquet loads 3-4x faster than CSV and preserves dtypes
matches_clean.to_parquet('matches_clean.parquet', index=False)
deliveries_full.to_parquet('deliveries_full.parquet', index=False)
print("Saved!")

Saved!


In [13]:
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency

# load clean data
matches_clean = pd.read_parquet('matches_clean.parquet')
deliveries_full = pd.read_parquet('deliveries_full.parquet')

In [14]:
# get season winners (last match of each season)
season_winners = matches_clean.groupby('season').apply(
    lambda x: x.loc[x['id'].idxmax(), 'winner']
).reset_index()
season_winners.columns = ['season','champion']

title_counts = season_winners['champion'].value_counts().reset_index()
title_counts.columns = ['team','titles']

fig1 = px.bar(title_counts, x='team', y='titles',
              title='IPL Titles by Team',
              color='titles', color_continuous_scale='teal',
              text='titles')
fig1.update_traces(textposition='outside')
fig1.update_layout(showlegend=False, xaxis_tickangle=-30)
fig1.show()

/tmp/ipykernel_898/2911326860.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  season_winners = matches_clean.groupby('season').apply(


In [15]:
# % of matches where toss winner also won the match
toss_impact = matches_clean['toss_match_win'].value_counts(normalize=True)*100
toss_impact = toss_impact.reset_index()
toss_impact.columns = ['won_toss_and_match','percentage']
toss_impact['won_toss_and_match'] = toss_impact['won_toss_and_match'].map(
    {True:'Won both', False:'Won toss, lost match'}
)

fig2 = px.pie(toss_impact, values='percentage', names='won_toss_and_match',
              title='Does Winning the Toss Help? (% of matches)',
              color_discrete_sequence=['#2196F3','#90CAF9'])
fig2.show()

print(f"Toss winner wins match: {toss_impact.iloc[0]['percentage']:.1f}% of the time")

Toss winner wins match: 50.8% of the time


In [16]:
# overall chasing win rate
chase_rate = matches_clean['chasing_team_won'].mean() * 100
print(f"Teams batting second win: {chase_rate:.1f}% of matches")

# by venue — top 10 venues
venue_chase = matches_clean.groupby('venue').agg(
    total=('winner','count'),
    chasing_wins=('chasing_team_won','sum')
).reset_index()
venue_chase['chase_pct'] = (venue_chase['chasing_wins']/venue_chase['total']*100).round(1)
venue_chase = venue_chase[venue_chase['total'] >= 10].sort_values('chase_pct', ascending=False).head(10)

fig3 = px.bar(venue_chase, x='venue', y='chase_pct',
              title='% Wins Batting Second — Top 10 Venues (min 10 matches)',
              color='chase_pct', color_continuous_scale='RdYlGn',
              text='chase_pct')
fig3.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig3.update_layout(xaxis_tickangle=-35)
fig3.show()

# chi-square test to validate the insight statistically
contingency = pd.crosstab(matches_clean['chasing_team_won'],
                           matches_clean['toss_decision'])
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square p-value: {p_value:.4f}")
print("Statistically significant!" if p_value < 0.05 else "Not significant")

Teams batting second win: 52.6% of matches



Chi-square p-value: 0.2831
Not significant


In [17]:
batsman_runs = deliveries_full.groupby('batter')['batsman_runs'].sum().reset_index()
batsman_runs.columns = ['batsman','total_runs']
top10 = batsman_runs.sort_values('total_runs', ascending=False).head(10)

fig4 = px.bar(top10, x='batsman', y='total_runs',
              title='Top 10 Run Scorers in IPL History',
              color='total_runs', color_continuous_scale='Blues',
              text='total_runs')
fig4.update_traces(textposition='outside')
fig4.update_layout(showlegend=False)
fig4.show()

In [18]:
# total runs scored each season
season_runs = deliveries_full.groupby('season')['total_runs'].sum().reset_index()
season_runs.columns = ['season','total_runs']

fig5 = px.line(season_runs, x='season', y='total_runs',
               title='Total Runs Scored Per Season',
               markers=True, line_shape='spline')
fig5.update_traces(line_color='#1D9E75', line_width=2.5)
fig5.show()